# SDS 05 — PageRank Fundamentals & Scalable PageRank Reference

Reusable reference notebook for the SDS comprehensive exam.

**Design goal:** understand the algorithm first, then have Spark-native patterns ready to adapt. No GraphFrames or third-party Spark packages are required.

The notebook contains:
- a pure-Python PageRank sanity implementation,
- dangling-node handling,
- convergence checks,
- Spark DataFrame PageRank for unweighted edges,
- weighted PageRank,
- Wikipedia `pageviews.csv` preparation,
- exam sanity checks and common traps.


## Recognition cue

- **global importance from directed links** → PageRank
- **topic/personalized importance** → change teleport vector (Lesson 6)
- **hubs and authorities** → HITS (Lesson 6)

Ordinary PageRank recurrence with uniform teleportation and dangling correction:

\[
r_{t+1}(v)=\frac{1-d}{N}+d\left(\sum_{u\to v}\frac{r_t(u)}{L(u)}+\frac{D_t}{N}\right)
\]

where \(D_t\) is total rank on dangling nodes.


In [ ]:
from collections import defaultdict

def pagerank_python(edges, d=0.85, tol=1e-12, max_iter=200):
    """Small learning/reference implementation. edges is iterable of (src,dst)."""
    edges = list(dict.fromkeys(edges))
    nodes = sorted(set([u for u,v in edges] + [v for u,v in edges]))
    N = len(nodes)
    out = defaultdict(list)
    for u,v in edges:
        out[u].append(v)
    r = {n: 1.0/N for n in nodes}
    for it in range(max_iter):
        dangling = sum(r[n] for n in nodes if len(out[n]) == 0)
        incoming = {n: 0.0 for n in nodes}
        for u in nodes:
            if out[u]:
                share = r[u] / len(out[u])
                for v in out[u]:
                    incoming[v] += share
        new = {n: (1-d)/N + d*(incoming[n] + dangling/N) for n in nodes}
        residual = sum(abs(new[n]-r[n]) for n in nodes)
        r = new
        if residual < tol:
            break
    return r, it+1, residual

edges_toy = [('A','B'),('A','C'),('B','C'),('C','A'),('C','D')]
ranks, iters, residual = pagerank_python(edges_toy)
print(ranks)
print('iterations:', iters, 'residual:', residual, 'sum:', sum(ranks.values()))
assert abs(sum(ranks.values()) - 1.0) < 1e-10


## One-iteration hand-check
For the toy graph `A→B, A→C, B→C, C→A, C→D`, node D is dangling. Starting at 0.25 each and `d=0.85`, one update gives:

- A = 0.196875
- B = 0.196875
- C = 0.409375
- D = 0.196875

The total is 1.0.


In [ ]:
def one_iteration_toy():
    d=0.85; N=4
    old={'A':.25,'B':.25,'C':.25,'D':.25}
    contrib={'A':.125,'B':.125,'C':.375,'D':.125}
    dangling=.25
    return {n:(1-d)/N + d*(contrib[n]+dangling/N) for n in old}
print(one_iteration_toy(), sum(one_iteration_toy().values()))


## Why the mass check matters
If dangling mass is ignored, rank disappears. Use `sum(rank) ≈ 1` as a cheap debugging check. The supplied Jan Problem 3a handled dangling mass explicitly; the comparison passing notebook did not.


In [ ]:
# Spark imports — run on the exam server / Spark environment
from pyspark.sql import functions as F
from pyspark.sql import DataFrame


## Reusable Spark PageRank function
This function assumes **one row per directed edge**. It precomputes transition probabilities and keeps the graph distributed. Only small scalars (`N`, dangling mass, residual, total mass) are returned to the driver.


In [ ]:
def pagerank_spark(edges: DataFrame, src='src', dst='dst', d=0.85, tol=1e-8, max_iter=50, checkpoint_every=0):
    # Canonical directed edge names
    edges = (edges.select(F.col(src).alias('src'), F.col(dst).alias('dst'))
                  .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
                  .dropDuplicates()
                  .cache())

    nodes = (edges.select(F.col('src').alias('id'))
                  .union(edges.select(F.col('dst').alias('id')))
                  .distinct()
                  .cache())
    N = nodes.count()
    if N == 0:
        raise ValueError('Graph has no nodes')

    outdeg = edges.groupBy('src').agg(F.count('*').alias('outdeg')).cache()

    # Static transition probabilities: compute once.
    trans = (edges.join(outdeg, 'src')
                  .select('src','dst',(F.lit(1.0)/F.col('outdeg')).alias('p'))
                  .cache())

    ranks = nodes.withColumn('rank', F.lit(1.0/N)).cache()
    history=[]

    for i in range(max_iter):
        dangling = (ranks.join(outdeg, ranks.id == outdeg.src, 'left_anti')
                         .agg(F.sum('rank').alias('dangling'))
                         .first()['dangling'])
        dangling = float(dangling or 0.0)

        incoming = (trans.join(ranks, trans.src == ranks.id, 'inner')
                         .select(F.col('dst').alias('id'),
                                 (F.col('rank')*F.col('p')).alias('contrib'))
                         .groupBy('id').agg(F.sum('contrib').alias('incoming')))

        new_ranks = (nodes.join(incoming,'id','left')
                          .fillna(0.0, subset=['incoming'])
                          .withColumn('rank',
                              F.lit((1.0-d)/N + d*dangling/N) + F.lit(d)*F.col('incoming'))
                          .select('id','rank')
                          .cache())

        residual = (new_ranks.alias('n').join(ranks.alias('o'),'id')
                     .agg(F.sum(F.abs(F.col('n.rank')-F.col('o.rank'))).alias('res'))
                     .first()['res'])
        residual=float(residual or 0.0)

        total_mass = new_ranks.agg(F.sum('rank').alias('mass')).first()['mass']
        history.append((i+1,residual,float(total_mass)))

        ranks.unpersist()
        ranks=new_ranks

        if checkpoint_every and (i+1) % checkpoint_every == 0:
            ranks = ranks.checkpoint(eager=True)

        if residual < tol:
            break

    return ranks, history


### Typical use
```python
ranks, hist = pagerank_spark(links_df, d=0.85, tol=1e-8, max_iter=50)
ranks.orderBy(F.desc('rank')).show(25, truncate=False)
print(hist[-1])  # iteration, residual, total mass
```

Expected sanity condition: total mass should remain close to 1.


## Wikipedia clickstream preparation — exam pattern
The supplied Problem 3 says to use `pageviews.csv` and consider only links. Both submitted notebooks treated the graph as unweighted unique `(src,dst)` pairs.


In [ ]:
def read_wikipedia_links(spark, path='pageviews.csv'):
    df = (spark.read.option('sep','\t').option('header',False).csv(path)
               .toDF('src','dst','type','count'))
    links = (df.filter(F.col('type')=='link')
               .select('src','dst')
               .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
               .dropDuplicates())
    return df, links

# df, links = read_wikipedia_links(spark)
# ranks, history = pagerank_spark(links)
# ranks.orderBy(F.desc('rank')).show(25, truncate=False)


## Weighted PageRank variant
If a future prompt explicitly says to use edge counts/traffic as weights, transition probability is

\[
p(u\to v)=\frac{w_{uv}}{\sum_z w_{uz}}.
\]


In [ ]:
def weighted_transition_edges(edges: DataFrame, weight='weight'):
    outw = edges.groupBy('src').agg(F.sum(F.col(weight)).alias('out_weight'))
    return (edges.join(outw,'src')
                 .withColumn('p', F.col(weight)/F.col('out_weight'))
                 .select('src','dst','p'))

def pagerank_spark_weighted(edges: DataFrame, weight='weight', d=0.85, tol=1e-8, max_iter=50):
    edges = (edges.select('src','dst',F.col(weight).cast('double').alias('weight'))
                  .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
                  .filter(F.col('weight') > 0)
                  .groupBy('src','dst').agg(F.sum('weight').alias('weight'))
                  .cache())
    nodes=(edges.select(F.col('src').alias('id')).union(edges.select(F.col('dst').alias('id'))).distinct().cache())
    N=nodes.count()
    trans=weighted_transition_edges(edges,'weight').cache()
    outsrc=edges.select('src').distinct().cache()
    ranks=nodes.withColumn('rank',F.lit(1.0/N)).cache()
    history=[]
    for i in range(max_iter):
        dangling=(ranks.join(outsrc,ranks.id==outsrc.src,'left_anti').agg(F.sum('rank').alias('s')).first()['s'])
        dangling=float(dangling or 0.0)
        incoming=(trans.join(ranks,trans.src==ranks.id,'inner')
                      .select(F.col('dst').alias('id'),(F.col('rank')*F.col('p')).alias('c'))
                      .groupBy('id').agg(F.sum('c').alias('incoming')))
        new=(nodes.join(incoming,'id','left').fillna(0.0,subset=['incoming'])
                  .withColumn('rank',F.lit((1-d)/N+d*dangling/N)+F.lit(d)*F.col('incoming'))
                  .select('id','rank').cache())
        residual=(new.alias('n').join(ranks.alias('o'),'id')
                    .agg(F.sum(F.abs(F.col('n.rank')-F.col('o.rank'))).alias('r')).first()['r'])
        residual=float(residual or 0.0)
        ranks.unpersist(); ranks=new
        history.append((i+1,residual))
        if residual<tol: break
    return ranks, history


## Exam checklist
Before submitting PageRank code, verify:

1. `src -> dst` direction is correct.
2. Destination-only nodes are included.
3. Source rank is divided by out-degree (or normalized edge weight).
4. Dangling mass is redistributed.
5. Damping/teleportation is explicit.
6. Large edges/ranks remain distributed.
7. Convergence or fixed-iteration approximation is stated.
8. `sum(rank)` is approximately 1.
9. Only requested top-k output is collected/displayed.


## Small critique drills

**Claim:** “PageRank of a page equals the sum of PageRank of pages it links to.”  
**Correction:** Wrong direction; rank arrives from incoming neighbors.

**Claim:** “Dangling nodes can be ignored because they do not point anywhere.”  
**Correction:** Their rank mass would disappear; redistribute it according to the teleport vector.

**Claim:** “Ten iterations is always enough.”  
**Correction:** Ten iterations may be a useful approximation, but convergence should be checked when possible.
